# Shallow Moment Equations: From INS to Solver

This notebook derives the **Shallow Moment Equations (SME)** step by step,
using the `DerivedModel` architecture. Each derivation step is a class
that can be inspected independently.

**Derivation graph (class hierarchy):**

```
FullINS                          -- 3D Navier-Stokes (root)
  |  apply(hydrostatic, Newtonian)
  |  depth_integrate + kinematic BCs
  v
SMEModel(DerivedModel)           -- solver-ready, projected onto basis
```

Each class is a node. The `derive_model()` method describes the edge.

## Part 1: The derivation, step by step

### 1.1 The full INS (root node)

`FullINS` builds the incompressible Navier-Stokes. Calling `describe()`
renders all equations as markdown — Jupyter displays it natively.

In [ ]:
from zoomy_core.model.models.ins_generator import (
    StateSpace, FullINS,
    KinematicBCBottom, KinematicBCSurface, HydrostaticPressure,
    Newtonian, Inviscid, DepthIntegrate, ZeroAtmosphericPressure,
)

state = StateSpace(dimension=2)
ins = FullINS(state)

# describe() returns a Description — Jupyter renders it as markdown
ins.describe()

With `strip_args=True`, function arguments are hidden for readability:

In [ ]:
ins.describe(strip_args=True)

### 1.2 Apply assumptions (the edge)

For the SME, we apply hydrostatic pressure and Newtonian viscosity.
Each assumption is a `Relation` — it has a name and substitution equations.

In [ ]:
hydro = HydrostaticPressure(state)
material = Newtonian(state)

# Assumptions render their substitution equations in notebooks
hydro

In [ ]:
material

### 1.3 Depth integration

Integrate each term over the water column $[b, b+h]$. The Leibniz rule
and fundamental theorem are applied term by term automatically.

At this stage, **boundary values remain** — we have not yet applied
kinematic BCs. This is intentional: the BCs are separate assumptions.

In [ ]:
z, b, eta = state.z, state.b, state.eta

# Apply assumptions to x-momentum
xm = ins.x_momentum.apply(HydrostaticPressure(state)).apply(Newtonian(state))

# Depth-integrate (NO kinematic BCs yet — boundary values remain)
di = DepthIntegrate(state)
xmom_raw = di.apply_to_equation(xm, state)
mass_raw = di.apply_to_equation(ins.continuity, state)

# Show with boundary values still present
xmom_raw.describe(strip_args=True)

### 1.4 Apply kinematic boundary conditions

Now substitute the kinematic BCs at the bottom and surface. These
are separate assumptions — each one renders its substitution equation.

In [ ]:
# The kinematic BCs are Assumptions — they show their substitution equations
kbc_s = KinematicBCSurface(state)
kbc_b = KinematicBCBottom(state)
print("Surface BC:"); display(kbc_s)
print("Bottom BC:"); display(kbc_b)

# Apply them
xmom = xmom_raw.apply(kbc_s).apply(kbc_b)
mass = mass_raw.apply(kbc_s).apply(kbc_b)

# Also set p_atm = 0
xmom = xmom.apply(ZeroAtmosphericPressure(state))
mass = mass.apply(ZeroAtmosphericPressure(state))

xmom.describe(strip_args=True)

### 1.5 Package as a DerivedSystem

The final depth-integrated equations with all assumptions applied:

In [ ]:
from zoomy_core.model.models.derived_system import DerivedSystem

system = DerivedSystem(
    "SME",
    {"mass": mass, "x_momentum": xmom},
    state,
    assumptions=["hydrostatic", "Newtonian", "depth_integrated",
                  "kinematic_bc_surface", "kinematic_bc_bottom", "p_atm=0"],
)
system.describe(strip_args=True)

You can toggle sections independently:

In [ ]:
# Header only, no equations
system.describe(final_equation=False, assumptions=False)

In [ ]:
# Equations + parameters, no header
system.describe(header=False, parameters=True, strip_args=True)

## Part 2: From derivation to solver

### 2.1 The class-based approach

`SMEModel` is a `DerivedModel` subclass. Its `derive_model()` method
IS the edge of the derivation graph — it specifies which assumptions
and operations transform the INS into the SME.

```python
class SMEModel(DerivedModel):
    projectable = True

    def derive_model(self):
        state = StateSpace(dimension=2)
        return sme(state, material=Newtonian(state))
```

The class itself is the node. `derive_model()` is the edge.
For a finer-grained graph, create intermediate classes.

In [ ]:
from zoomy_core.model.models.sme_model import SMEModel

model = SMEModel(level=1, eigenvalue_mode="numerical")

# DerivedModel.describe() delegates to its DerivedSystem
model.describe(strip_args=True)

### 2.2 Chaining intermediate models

For a finer-grained derivation graph, use intermediate classes.
Each class is a node you can inspect independently:

In [ ]:
from zoomy_core.model.models.derived_model import DerivedModel
from zoomy_core.model.models.sme_model import INSModel

# Intermediate: hydrostatic + depth-integrated, but no material or BCs
class DepthIntegratedINS(INSModel):
    def derive_model(self):
        super().derive_model()
        self.apply(HydrostaticPressure(self.state))
        self.apply(DepthIntegrate(self.state))

# Final: add Newtonian, kinematic BCs, p_atm=0, and project
class ViscousSME(DepthIntegratedINS):
    projectable = True
    def derive_model(self):
        super().derive_model()
        self.apply(KinematicBCSurface(self.state))
        self.apply(KinematicBCBottom(self.state))
        self.apply(Newtonian(self.state))
        self.apply(ZeroAtmosphericPressure(self.state))

# Inspect the intermediate — still has boundary values!
intermediate = DepthIntegratedINS()
print(f"Intermediate (projectable={intermediate.projectable}):")
intermediate.describe(strip_args=True)

Now the final model — solver-ready, all functions compiled:

In [ ]:
final = ViscousSME(level=1, eigenvalue_mode="numerical")

# Derivation graph: HydrostaticSME → ViscousSME (compact)
final.describe(derivation='mermaid', strip_args=True, final_equation=False)

In [ ]:
# Same graph as markdown — shows equations at each node + operation details on edges
final.describe(derivation='markdown', strip_args=True, final_equation=False)

In [ ]:
# Just the final result — no derivation path
final.describe(strip_args=True)

## Part 3: Dam-break simulation

### 3.1 Setup

In [ ]:
import numpy as np
import zoomy_core.model.boundary_conditions as BC
import zoomy_core.model.initial_conditions as IC
from zoomy_core.mesh import BaseMesh
from zoomy_core.fvm.solver_imex_numpy import FSFIMEXSolver
import zoomy_core.fvm.timestepping as ts

# Use the one-shot SMEModel for the simulation
model = SMEModel(level=1, eigenvalue_mode="numerical")
n_vars = model.n_variables

model.initial_conditions = IC.RP(
    jump_position_x=5.0,
    high=lambda n: np.array([0.0, 1.0] + [0.0] * (n_vars - 2)),
    low=lambda n: np.array([0.0, 0.5] + [0.0] * (n_vars - 2)),
)
model.boundary_conditions = BC.BoundaryConditions(
    boundary_conditions_list=[
        BC.Extrapolation(tag="left"),
        BC.Extrapolation(tag="right"),
    ]
)

mesh = BaseMesh.create_1d((0, 10), n_inner_cells=200)
solver = FSFIMEXSolver(time_end=1.0, compute_dt=ts.adaptive(CFL=0.4))
Q, _ = solver.solve(mesh, model, write_output=False)

### 3.2 Results

In [ ]:
from zoomy_core.mesh import ensure_lsq_mesh
import matplotlib.pyplot as plt

lsq = ensure_lsq_mesh(mesh, model)
nc = lsq.n_inner_cells
x = lsq.cell_centers[0, :nc]
h = Q[1, :nc]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(x, h, "b-", lw=1.5)
axes[0].set_xlabel("x"); axes[0].set_ylabel("h"); axes[0].set_title("Water depth")
axes[0].grid(True, alpha=0.3)

u0 = np.where(h > 1e-6, Q[2, :nc] / h, 0.0)
axes[1].plot(x, u0, "r-", lw=1.5)
axes[1].set_xlabel("x"); axes[1].set_ylabel("$u_0$"); axes[1].set_title("Mean velocity")
axes[1].grid(True, alpha=0.3)

u1 = np.where(h > 1e-6, Q[3, :nc] / h, 0.0)
axes[2].plot(x, u1, "g-", lw=1.5)
axes[2].set_xlabel("x"); axes[2].set_ylabel("$\\alpha_1$"); axes[2].set_title("Profile mode 1")
axes[2].grid(True, alpha=0.3)

fig.suptitle(f"SME Dam Break (level={model.level}, t={solver.time_end})", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
### 3.3 Level comparison (SWE vs SME level 1 vs level 2)

results = {}
for level in [0, 1, 2]:
    m = SMEModel(level=level, eigenvalue_mode="numerical")
    nv = m.n_variables
    m.initial_conditions = IC.RP(
        jump_position_x=5.0,
        high=lambda n, nv=nv: np.array([0.0, 1.0] + [0.0] * (nv - 2)),
        low=lambda n, nv=nv: np.array([0.0, 0.5] + [0.0] * (nv - 2)),
    )
    m.boundary_conditions = BC.BoundaryConditions(
        boundary_conditions_list=[BC.Extrapolation(tag="left"), BC.Extrapolation(tag="right")]
    )
    msh = BaseMesh.create_1d((0, 10), 200)
    Q_out, _ = FSFIMEXSolver(time_end=1.0, compute_dt=ts.adaptive(CFL=0.4)).solve(msh, m, write_output=False)
    lsq_m = ensure_lsq_mesh(msh, m)
    results[level] = {"x": lsq_m.cell_centers[0, :200], "h": Q_out[1, :200], "hu0": Q_out[2, :200]}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for lvl, r in results.items():
    ax1.plot(r["x"], r["h"], lw=1.5, label=f"level {lvl}")
    ax2.plot(r["x"], np.where(r["h"] > 1e-6, r["hu0"] / r["h"], 0), lw=1.5, label=f"level {lvl}")
ax1.set_xlabel("x"); ax1.set_ylabel("h"); ax1.set_title("Water depth"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_xlabel("x"); ax2.set_ylabel("$u_0$"); ax2.set_title("Mean velocity"); ax2.legend(); ax2.grid(True, alpha=0.3)
fig.suptitle("Level comparison (t = 1.0)", fontsize=13)
plt.tight_layout()
plt.show()